# JSON Preparation

In [1]:
import json
from pathlib import Path

import requests


# Minimum working request
url = "https://archive-api.open-meteo.com/v1/archive"

params = {
    "latitude": 51.0447,
    "longitude": -114.0719,
    "start_date": "2025-01-01",
    "end_date": "2025-01-02",
    "hourly": "temperature_2m,precipitation",
    "models": "era5",
    "timezone": "GMT",
    "timeformat": "unixtime",
}

response = requests.get(
    url,
    params=params,
    timeout=60,
)
response.raise_for_status()

payload = response.json()

payload.keys(), payload["hourly"].keys()


(dict_keys(['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'hourly_units', 'hourly']),
 dict_keys(['time', 'temperature_2m', 'precipitation']))

In [2]:
# JSON keeps structure and basic scalar types.
sample_path = Path("sample_weather.json")

with sample_path.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        payload,
        file,
        allow_nan=False,
    )

with sample_path.open(
    "r",
    encoding="utf-8",
) as file:
    sample = json.load(file)

print(sample.keys())
print(sample["hourly_units"])
print(sample["hourly"]["time"][:3])

sample_path.unlink()


dict_keys(['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'hourly_units', 'hourly'])
{'time': 'unixtime', 'temperature_2m': '°C', 'precipitation': 'mm'}
[1735689600, 1735693200, 1735696800]


In [3]:
import time

import pandas as pd


# Configuration
URL = "https://archive-api.open-meteo.com/v1/archive"

LATITUDE = 51.0447
LONGITUDE = -114.0719
MODEL = "era5"
LOCAL_TIMEZONE = "America/Edmonton"

HOURLY_VARIABLES = [
    "temperature_2m",
    "relative_humidity_2m",
    "dew_point_2m",
    "precipitation",
    "rain",
    "snowfall",
    "weather_code",
    "cloud_cover",
    "pressure_msl",
    "surface_pressure",
    "wind_speed_10m",
    "wind_direction_10m",
    "shortwave_radiation",
    "vapour_pressure_deficit",
    "soil_temperature_0_to_7cm",
]

COLUMNS = ["time", *HOURLY_VARIABLES]

START_LOCAL = pd.Timestamp(
    "2000-01-01 00:00",
    tz=LOCAL_TIMEZONE,
)

END_LOCAL = pd.Timestamp(
    "2026-10-01 00:00",
    tz=LOCAL_TIMEZONE,
)

REQUEST_DELAY_SECONDS = 4

DATA_PATH = Path("data") / "weather.json"


In [4]:
def create_year_batches(
    start_local,
    end_local,
):
    batches = []
    batch_start = start_local

    while batch_start < end_local:
        next_year = pd.Timestamp(
            year=batch_start.year + 1,
            month=1,
            day=1,
            tz=batch_start.tz,
        )

        batch_end = min(
            next_year,
            end_local,
        )

        batches.append(
            (batch_start, batch_end)
        )

        batch_start = batch_end

    return batches


In [5]:
def fetch_weather_batch(
    start_local,
    end_local,
):
    start_utc = start_local.tz_convert(
        "UTC"
    )
    end_utc = end_local.tz_convert(
        "UTC"
    )

    # API accepts dates, so request the UTC envelope then trim.
    params = {
        "latitude": LATITUDE,
        "longitude": LONGITUDE,
        "start_date": (
            start_utc.date().isoformat()
        ),
        "end_date": (
            end_utc
            - pd.Timedelta(hours=1)
        ).date().isoformat(),
        "hourly": ",".join(
            HOURLY_VARIABLES
        ),
        "models": MODEL,
        "timezone": "GMT",
        "timeformat": "unixtime",
    }

    response = requests.get(
        URL,
        params=params,
        timeout=60,
    )
    response.raise_for_status()

    payload = response.json()

    metadata = {
        "latitude": payload["latitude"],
        "longitude": payload["longitude"],
        "elevation": payload["elevation"],
        "timezone": payload["timezone"],
        "utc_offset_seconds": (
            payload["utc_offset_seconds"]
        ),
        "hourly_units": (
            payload["hourly_units"]
        ),
    }

    hourly = payload["hourly"]

    time_index = pd.to_datetime(
        hourly["time"],
        unit="s",
        utc=True,
    )

    keep = (
        (time_index >= start_utc)
        & (time_index < end_utc)
    )

    hourly = {
        key: [
            value
            for value, include
            in zip(values, keep)
            if include
        ]
        for key, values
        in hourly.items()
    }

    return metadata, hourly


In [6]:
batch_metadata, batch_hourly = (
    fetch_weather_batch(
        pd.Timestamp(
            "2025-01-01 00:00",
            tz=LOCAL_TIMEZONE,
        ),
        pd.Timestamp(
            "2025-01-03 00:00",
            tz=LOCAL_TIMEZONE,
        ),
    )
)

print(batch_metadata)
print(len(batch_hourly["time"]))

{'latitude': 51.0, 'longitude': -114.0, 'elevation': 1051.0, 'timezone': 'GMT', 'utc_offset_seconds': 0, 'hourly_units': {'time': 'unixtime', 'temperature_2m': '°C', 'relative_humidity_2m': '%', 'dew_point_2m': '°C', 'precipitation': 'mm', 'rain': 'mm', 'snowfall': 'cm', 'weather_code': 'wmo code', 'cloud_cover': '%', 'pressure_msl': 'hPa', 'surface_pressure': 'hPa', 'wind_speed_10m': 'km/h', 'wind_direction_10m': '°', 'shortwave_radiation': 'W/m²', 'vapour_pressure_deficit': 'kPa', 'soil_temperature_0_to_7cm': '°C'}}
48


In [7]:
def fetch_weather(
    start_local,
    end_local,
):
    batches = create_year_batches(
        start_local,
        end_local,
    )

    metadata = None

    hourly = {
        column: []
        for column in COLUMNS
    }

    for i, (
        batch_start,
        batch_end,
    ) in enumerate(
        batches,
        start=1,
    ):
        print(
            f"Request {i}/{len(batches)}: "
            f"{batch_start} -> {batch_end}"
        )

        (
            batch_metadata,
            batch_hourly,
        ) = fetch_weather_batch(
            batch_start,
            batch_end,
        )

        if metadata is None:
            metadata = batch_metadata
        elif batch_metadata != metadata:
            raise ValueError(
                "Source metadata changed "
                "across batches."
            )

        for column in COLUMNS:
            hourly[column].extend(
                batch_hourly[column]
            )

        if i < len(batches):
            time.sleep(
                REQUEST_DELAY_SECONDS
            )

    expected_time = pd.date_range(
        start=start_local.tz_convert(
            "UTC"
        ),
        end=end_local.tz_convert(
            "UTC"
        ),
        freq="h",
        inclusive="left",
    )

    actual_time = pd.DatetimeIndex(
        pd.to_datetime(
            hourly["time"],
            unit="s",
            utc=True,
        )
    )

    if not actual_time.equals(
        expected_time
    ):
        raise ValueError(
            "Unexpected hourly coverage."
        )

    if {
        len(values)
        for values in hourly.values()
    } != {len(expected_time)}:
        raise ValueError(
            "Unexpected hourly array "
            "lengths."
        )

    return {
        **metadata,
        "hourly": hourly,
    }


In [8]:
# Cross-year merge check
test_weather = fetch_weather(
    pd.Timestamp(
        "2024-12-31 00:00",
        tz=LOCAL_TIMEZONE,
    ),
    pd.Timestamp(
        "2025-01-02 00:00",
        tz=LOCAL_TIMEZONE,
    ),
)

print(test_weather.keys())
print(
    len(
        test_weather[
            "hourly"
        ]["time"]
    )
)


Request 1/2: 2024-12-31 00:00:00-07:00 -> 2025-01-01 00:00:00-07:00
Request 2/2: 2025-01-01 00:00:00-07:00 -> 2025-01-02 00:00:00-07:00
dict_keys(['latitude', 'longitude', 'elevation', 'timezone', 'utc_offset_seconds', 'hourly_units', 'hourly'])
48


In [9]:
# Full preparation
weather = fetch_weather(
    START_LOCAL,
    END_LOCAL,
)

DATA_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

with DATA_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        weather,
        file,
        allow_nan=False,
    )

print(
    f"Rows: "
    f"{len(weather['hourly']['time']):,}"
)
print(f"Saved: {DATA_PATH}")


Request 1/27: 2000-01-01 00:00:00-07:00 -> 2001-01-01 00:00:00-07:00
Request 2/27: 2001-01-01 00:00:00-07:00 -> 2002-01-01 00:00:00-07:00
Request 3/27: 2002-01-01 00:00:00-07:00 -> 2003-01-01 00:00:00-07:00
Request 4/27: 2003-01-01 00:00:00-07:00 -> 2004-01-01 00:00:00-07:00
Request 5/27: 2004-01-01 00:00:00-07:00 -> 2005-01-01 00:00:00-07:00
Request 6/27: 2005-01-01 00:00:00-07:00 -> 2006-01-01 00:00:00-07:00
Request 7/27: 2006-01-01 00:00:00-07:00 -> 2007-01-01 00:00:00-07:00
Request 8/27: 2007-01-01 00:00:00-07:00 -> 2008-01-01 00:00:00-07:00
Request 9/27: 2008-01-01 00:00:00-07:00 -> 2009-01-01 00:00:00-07:00
Request 10/27: 2009-01-01 00:00:00-07:00 -> 2010-01-01 00:00:00-07:00
Request 11/27: 2010-01-01 00:00:00-07:00 -> 2011-01-01 00:00:00-07:00
Request 12/27: 2011-01-01 00:00:00-07:00 -> 2012-01-01 00:00:00-07:00
Request 13/27: 2012-01-01 00:00:00-07:00 -> 2013-01-01 00:00:00-07:00
Request 14/27: 2013-01-01 00:00:00-07:00 -> 2014-01-01 00:00:00-07:00
Request 15/27: 2014-01-01 00:

In [10]:
def main():
    weather = fetch_weather(
        START_LOCAL,
        END_LOCAL,
    )

    DATA_PATH.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    with DATA_PATH.open(
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            weather,
            file,
            allow_nan=False,
        )

    print(
        f"Rows: "
        f"{len(weather['hourly']['time']):,}"
    )
    print(f"Saved: {DATA_PATH}")
